[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pflahert/hd-stats-workshop/blob/main/notebooks/lab2.ipynb)

# Lab 2: The Histogram That Was Too Wide
## From FDR Control to Empirical Bayes

**Day 1, 11:00–12:00** | Learning Outcomes: LO1, LO3

This lab applies both Day-1 lectures to the Golub dataset:

- **From Lecture 1 (testing)**: implement the BH step-up procedure by hand, estimate $\pi_0$ with Storey's method, and compute $q$-values manually.
- **From Lecture 2 (estimation)**: shrink gene-level variances with a limma-style moderated $t$-statistic, examine the empirical null distribution, build a volcano plot, compare methods on the same dataset, and (with AI help) compute Efron's local FDR.

Lecture 1 told you what BH and $q$-values *do*. Lecture 2 told you how to read the same histogram as an *estimation* object — a mixture $f = \pi_0 f_0 + (1-\pi_0)f_1$ that produces a per-test posterior probability of being null. This lab exercises both perspectives, on the Welch $t$-tests computed below.

---

**How this lab uses AI**

This lab integrates AI at several steps, not just at the end. Two kinds of AI exercises appear:

- **Single-shot critiques** — you prompt, paste the response, and evaluate it against a named checklist (Exercise 1.2, Section 7).
- **Multi-turn conversations** — you prompt, critique the response, write a follow-up prompt that addresses the specific weakness you flagged, and continue *in the same chat session* so the AI can revise its earlier answer. *At least one true multi-turn exercise per lab is required.* In this lab, **Exercise 4.2 (empirical null)** is the multi-turn one.

Use any AI assistant (the [UMass campus GenAI platform](https://www.umass.edu/it/genai/platform), Claude, ChatGPT, or another). Note which AI you used for each prompt.

---


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests

# Workshop convention: deterministic seed.
np.random.seed(2026)

# Surface RuntimeWarnings from norm.ppf(1) = inf, zero-variance genes, etc. —
# the empirical-null section needs students to see these.

try:
    plt.style.use('seaborn-v0_8-whitegrid')
except OSError:
    plt.style.use('default')

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12


In [ ]:
# Load the Golub expression matrix and metadata from the workshop repo (CSV — same source as Labs 1, 3).
expr_url = "https://raw.githubusercontent.com/pflahert/hd-stats-workshop/v2026-spring-data/data/golub_expression.csv"
meta_url = "https://raw.githubusercontent.com/pflahert/hd-stats-workshop/v2026-spring-data/data/golub_metadata.csv"

expr_genes_x_samples = pd.read_csv(expr_url, index_col=0)
meta = pd.read_csv(meta_url)

assert expr_genes_x_samples.shape == (3051, 72), \
    f"Unexpected expression shape: {expr_genes_x_samples.shape} (expected (3051, 72))"
assert list(expr_genes_x_samples.columns) == list(meta["sample_id"]), \
    "Sample IDs do not match between expression and metadata."

expr = expr_genes_x_samples.T                       # samples (rows) x genes (cols)
labels = meta["subtype"].values
all_mask = labels == "ALL"
aml_mask = labels == "AML"

print(f"Expression matrix: {expr.shape[0]} samples x {expr.shape[1]} genes")
print(f"Groups: ALL={int(all_mask.sum())}, AML={int(aml_mask.sum())}")
print(f"Ratio p/n: {expr.shape[1] / expr.shape[0]:.1f}")

# Drop genes that are near-constant within either group before testing — these
# carry no information for differential expression and scipy's t-test would emit
# a "precision loss in moment calculation" RuntimeWarning on them. Standard
# preprocessing step in real microarray / RNA-seq pipelines.
gene_names_full = expr.columns.values
all_block_full = expr.loc[all_mask].values.astype(float)
aml_block_full = expr.loc[aml_mask].values.astype(float)

MIN_GROUP_VAR = 1e-8
informative = (all_block_full.var(axis=0) > MIN_GROUP_VAR) & \
              (aml_block_full.var(axis=0) > MIN_GROUP_VAR)
n_dropped = int((~informative).sum())
if n_dropped:
    print(f"Dropped {n_dropped} gene(s) with near-constant expression in one or both groups.")

all_block = all_block_full[:, informative]
aml_block = aml_block_full[:, informative]
gene_names = gene_names_full[informative]
n_genes = len(gene_names)

# Vectorized per-gene Welch t-tests on the samples-as-rows matrix.
t_stats, pvals = stats.ttest_ind(all_block, aml_block, equal_var=False, axis=0)
log2fc = all_block.mean(axis=0) - aml_block.mean(axis=0)   # mean difference (approx log2 FC on log-scale data)

print(f"\nComputed p-values for {len(pvals)} genes")
print(f"Significant at alpha = 0.05 (uncorrected): {int(np.sum(pvals < 0.05))}")

# p-value histogram
fig, ax = plt.subplots()
ax.hist(pvals, bins=100, density=True, color='steelblue', edgecolor='white', alpha=0.8)
ax.axhline(y=1, color='red', linestyle='--', linewidth=2, label='Uniform(0,1)')
ax.set_xlabel('p-value')
ax.set_ylabel('Density')
ax.set_title('p-value histogram: ALL vs AML (Welch t-tests)')
ax.legend()
plt.tight_layout()
plt.show()


---

## Part 1: BH Step-Up Procedure (By Hand)

You're going to implement the Benjamini-Hochberg procedure yourself. The idea is beautifully geometric:

1. Sort the p-values from smallest to largest: $p_{(1)} \le p_{(2)} \le \cdots \le p_{(m)}$
2. Compare each sorted p-value to its BH threshold line: $\frac{i}{m} \cdot \alpha$
3. Find the largest $k$ where $p_{(k)} \le \frac{k}{m} \cdot \alpha$
4. Reject hypotheses $1, 2, \ldots, k$

This is the step-up procedure from Lecture 1. Implement it below and verify it matches `statsmodels`.

In [ ]:
alpha = 0.05
m = len(pvals)

# Step 1: Sort p-values and keep track of original indices
sorted_indices = np.argsort(pvals)
sorted_pvals = pvals[sorted_indices]

# Step 2: Compute BH threshold for each rank
ranks = np.arange(1, m + 1)
bh_threshold = (ranks / m) * alpha

# Step 3: Find the largest k where p_(k) <= (k/m) * alpha
below_line = sorted_pvals <= bh_threshold
if np.any(below_line):
    k_bh = int(np.max(np.where(below_line)[0])) + 1   # +1: 0-indexed -> rank
else:
    k_bh = 0

print(f"BH procedure (hand-coded): {k_bh} discoveries at FDR <= {alpha}")

# Verify against statsmodels: compare the actual rejection *index sets*, not just counts.
reject_sm, padj_sm, _, _ = multipletests(pvals, alpha=alpha, method='fdr_bh')
reject_hand = np.zeros(m, dtype=bool)
reject_hand[sorted_indices[:k_bh]] = True
print(f"BH procedure (statsmodels): {int(np.sum(reject_sm))} discoveries at FDR <= {alpha}")
assert np.array_equal(reject_hand, reject_sm), \
    "Hand-coded and statsmodels BH disagree on which genes are rejected."
print("Rejection sets match exactly.")

# Plot: sorted p-values vs BH line
fig, ax = plt.subplots(figsize=(10, 6))
n_plot = min(2000, m)
ax.scatter(ranks[:n_plot], sorted_pvals[:n_plot], s=1, alpha=0.5, color='steelblue',
           label='Sorted p-values')
ax.plot(ranks[:n_plot], bh_threshold[:n_plot], color='red', linewidth=2,
        label=f'BH line (slope = {alpha}/m)')
ax.axvline(x=k_bh, color='green', linestyle='--', linewidth=1.5,
           label=f'k* = {k_bh} (cutoff)')
ax.set_xlabel('Rank (i)')
ax.set_ylabel('p-value')
ax.set_title('BH Step-Up Procedure: Sorted p-values vs. Threshold Line')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()


**Exercise 1.1**: Your hand-coded BH and `statsmodels` should agree (the cell asserts the rejection *sets* match, not just the counts). How many discoveries at FDR $\le 0.05$?

**YOUR ANSWER:**


**Exercise 1.2 — AI implements BH on a worked example (single-shot)**

A surprisingly common AI error in BH is to use the *first* crossing of the threshold line instead of the *last*. Ask an AI to demonstrate the procedure on a small worked example and check whether it gets this detail right. **The prompt below deliberately *does not* tell the AI which crossing to use — that is the test.**

> "Reply with a single Python code block (no prose, no .docx file — just runnable code I can paste into a Colab cell). Implement the Benjamini–Hochberg procedure at $\alpha = 0.10$ on this small worked example with $m = 10$ sorted p-values: `[0.001, 0.008, 0.039, 0.041, 0.042, 0.060, 0.074, 0.205, 0.212, 0.500]`. Print a table with columns: rank $i$, $p_{(i)}$, BH threshold $i\alpha/m$, below-line?. Then print the final discovery count $k$ and explain in one sentence which rule you used to pick $k$. Use `numpy` only."

**AI's response (paste here):**


**Critique checklist:**

| Criterion | Pass / Fail / Partial | Notes |
|---|---|---|
| Did the AI use the *largest* crossing (step-up), not the first crossing (step-down)? | | |
| Did it compute the threshold as $i\alpha/m$ (not $\alpha/(im)$ or some transposition)? | | |
| Did it reject *all* hypotheses with rank $\le k$, not just the index that crossed? | | |
| Is the output table correctly formatted, with all columns labeled? | | |
| Did the AI explain *which* rule it used (largest $k$)? | | |

**Correct answer** for this example: $i\alpha/m$ values are $0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.10$. Comparing $p_{(i)}$ to its threshold, the crossings (where $p_{(i)} \le i\alpha/m$) are at ranks 1, 2, and 5. The *largest* crossing is at rank 5, so $k = 5$ and BH rejects the first 5 hypotheses.

**One-sentence overall assessment** — did the AI get the "largest $k$" detail correct, or did it return the step-down answer ($k = 2$)?

---

## Part 2: Storey's $\pi_0$ Estimator (By Hand)

The BH procedure implicitly assumes all hypotheses could be null ($\pi_0 = 1$). But look at your p-value histogram: many genes clearly have signal. If we estimate the fraction of true nulls $\pi_0$, we can be more powerful.

**Key insight**: For any threshold $\lambda$, p-values above $\lambda$ are mostly from true nulls (since null p-values are uniform). So:

$$\hat{\pi}_0(\lambda) = \frac{\#\{p_i > \lambda\}}{m(1 - \lambda)}$$

We compute this for a range of $\lambda$ values and pick a stable estimate.

In [ ]:
def estimate_pi0(pvals, lambda_range=np.arange(0.05, 0.96, 0.05)):
    """
    Estimate pi0 using Storey's method.

    For each lambda, estimate pi0 as the fraction of p-values above lambda,
    divided by (1 - lambda). Then select a stable estimate by taking the
    *median* of the estimates in the stable middle range (lambda in [0.4, 0.9]) —
    the very largest lambda values use too few p-values in the numerator and
    are noisy.

    Storey's qvalue R package fits a smooth spline of pi0(lambda) and reads off
    its limit as lambda -> 1. We use a simpler middle-range median here and
    cap the result at 1.0.
    """
    m = len(pvals)
    pi0_estimates = np.array([
        np.sum(pvals > lam) / (m * (1 - lam))
        for lam in lambda_range
    ])
    # Stable plateau: average over lambda in [0.4, 0.9]
    mid_mask = (lambda_range >= 0.4) & (lambda_range <= 0.9)
    pi0_hat = float(np.median(pi0_estimates[mid_mask]))
    pi0_hat = min(pi0_hat, 1.0)

    return pi0_hat, pi0_estimates, lambda_range


pi0_hat, pi0_estimates, lambda_vals = estimate_pi0(pvals)

print(f"Estimated pi0: {pi0_hat:.4f}")
print(f"Interpretation: ~{pi0_hat*100:.1f}% of genes are estimated to be truly null")
print(f"              ~{(1-pi0_hat)*100:.1f}% of genes show a real difference between ALL and AML")

# Plot pi0 estimates vs lambda
fig, ax = plt.subplots()
ax.plot(lambda_vals, pi0_estimates, 'o-', color='steelblue', markersize=5,
        label=r'$\hat{\pi}_0(\lambda)$')
ax.axvspan(0.4, 0.9, color='lightyellow', alpha=0.5, label=r'stable range used for median')
ax.axhline(y=pi0_hat, color='red', linestyle='--', linewidth=1.5,
           label=f'$\\hat{{\\pi}}_0$ = {pi0_hat:.3f}')
ax.set_xlabel(r'$\lambda$')
ax.set_ylabel(r'$\hat{\pi}_0(\lambda)$')
ax.set_title("Storey's pi_0 estimate vs lambda — middle-range median used")
ax.set_ylim(0, 1.05)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()


**Exercise 2.1**: What is your estimate of $\pi_0$? What does it mean biologically -- roughly what fraction of genes show no difference between ALL and AML?

**YOUR ANSWER:**

---

---

## Part 2.5: limma-style Moderated $t$ (Empirical-Bayes Variance Shrinkage)

Lecture 2's central practical instance of empirical Bayes is `limma`'s moderated $t$-statistic ([Smyth 2004](https://doi.org/10.2202/1544-6115.1027)). The motivating problem: with $n_A = 47$ and $n_B = 25$, the per-gene sample variance $s_g^2$ is itself noisy, and a gene whose true variance happens to be small can produce a coincidentally tiny $s_g$ — and therefore a spuriously large raw $t$-statistic.

The fix is to shrink each gene's variance estimate toward a population prior:

$$\tilde s_g^2 = \frac{d_0\, s_0^2 + d_g\, s_g^2}{d_0 + d_g}, \qquad \tilde t_g = \frac{\bar y_{g,A} - \bar y_{g,B}}{\tilde s_g \sqrt{1/n_A + 1/n_B}}.$$

Under Smyth's full procedure $(d_0, s_0^2)$ are fit by trigamma matching to the population of gene-level log-variances; we use the simpler heuristic $d_0 = 4$, $s_0^2 = \exp(\overline{\log s_g^2})$. The reference distribution under the null is $t_{d_0 + d_g}$.

In [ ]:
# Smyth-style moderated t on the Golub data.
d_g = all_block.shape[0] + aml_block.shape[0] - 2   # 47 + 25 - 2 = 70
# Per-gene pooled sample variance s_g^2 (matches the Welch t in cell-2 closely enough for the shrinkage demonstration).
ss_within = ((all_block - all_block.mean(axis=0))**2).sum(axis=0)           + ((aml_block - aml_block.mean(axis=0))**2).sum(axis=0)
s_g_sq = ss_within / d_g

# Simple plug-in prior: d_0 = 4, s_0^2 = exp(mean log s_g^2).
d_0 = 4.0
s_0_sq = float(np.exp(np.mean(np.log(s_g_sq))))
s_tilde_sq = (d_0 * s_0_sq + d_g * s_g_sq) / (d_0 + d_g)

mean_diff = all_block.mean(axis=0) - aml_block.mean(axis=0)
t_mod = mean_diff / np.sqrt(s_tilde_sq * (1.0/all_block.shape[0] + 1.0/aml_block.shape[0]))
p_mod = 2 * (1 - stats.t.cdf(np.abs(t_mod), df=d_0 + d_g))

reject_mod, _, _, _ = multipletests(p_mod, alpha=0.05, method='fdr_bh')
n_raw_bh = int(np.sum(reject_sm))
n_mod_bh = int(np.sum(reject_mod))

# Shrinkage scatter: raw s_g vs moderated s_tilde
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
s_g = np.sqrt(s_g_sq); s_tilde = np.sqrt(s_tilde_sq)
mx = float(np.percentile(np.concatenate([s_g, s_tilde]), 99))
axes[0].plot([0, mx], [0, mx], color="#888", lw=1, linestyle="--", label="no shrinkage")
axes[0].axhline(np.sqrt(s_0_sq), color="#c0504d", lw=1.2, linestyle=":",
                label=fr"prior $\sqrt{{s_0^2}} \approx {np.sqrt(s_0_sq):.2f}$")
axes[0].scatter(s_g, s_tilde, s=4, alpha=0.5, color="#2a4d9c")
axes[0].set_xlabel(r"raw $s_g$"); axes[0].set_ylabel(r"moderated $\tilde s_g$")
axes[0].set_xlim(0, mx); axes[0].set_ylim(0, mx)
axes[0].set_title(rf"variance shrinkage ($d_0={d_0:.0f}$)")
axes[0].legend(loc="upper left", fontsize=9, frameon=False)

axes[1].bar(['raw t + BH', 'moderated t + BH'], [n_raw_bh, n_mod_bh],
            color=['#888', '#c0504d'])
axes[1].set_ylabel('discoveries at FDR <= 0.05')
axes[1].set_title('raw vs moderated discovery counts')
for i, v in enumerate([n_raw_bh, n_mod_bh]):
    axes[1].text(i, v + max(n_raw_bh, n_mod_bh)*0.01, str(v),
                 ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

print(f"Raw Welch-t + BH discoveries:    {n_raw_bh}")
print(f"Moderated-t + BH discoveries:    {n_mod_bh}")


**Exercise 2.5.1**: Compare the raw-t vs moderated-t discovery counts above. The moderated-t test is typically more conservative on genes with anomalously small $s_g$ (because $\tilde s_g > s_g$ there) and more sensitive on genes with anomalously large $s_g$. Did the net discovery count go up, down, or stay roughly the same on this dataset? What does that tell you about the variance landscape of the Golub data?

**YOUR ANSWER:**

---

## Part 3: Q-Values (Manual Computation)

The **q-value** of a gene is the minimum FDR at which that gene would be called significant. It incorporates the $\pi_0$ estimate, making it more powerful than BH when many genes are truly null.

The q-value for the gene with the $i$-th smallest p-value is:

$$q_{(i)} = \min_{j \ge i} \frac{\hat{\pi}_0 \cdot m \cdot p_{(j)}}{j}$$

We compute this from the bottom up, taking cumulative minima.

In [ ]:
def compute_qvalues(pvals, pi0):
    """
    Compute q-values given p-values and an estimate of pi0.
    
    Parameters
    ----------
    pvals : array-like
        Raw p-values.
    pi0 : float
        Estimated proportion of true nulls.
    
    Returns
    -------
    qvals : array
        Q-values in the same order as the input p-values.
    """
    m = len(pvals)
    sorted_indices = np.argsort(pvals)
    sorted_pvals = pvals[sorted_indices]
    
    # Compute raw q-values: pi0 * m * p_(i) / i
    ranks = np.arange(1, m + 1)
    raw_qvals = pi0 * m * sorted_pvals / ranks
    
    # Enforce monotonicity: q_(i) <= q_(i+1) by taking cumulative min from the right
    qvals_sorted = np.minimum.accumulate(raw_qvals[::-1])[::-1]
    qvals_sorted = np.minimum(qvals_sorted, 1.0)  # cap at 1
    
    # Map back to original order
    qvals = np.empty(m)
    qvals[sorted_indices] = qvals_sorted
    
    return qvals


qvals = compute_qvalues(pvals, pi0_hat)

# Compare discovery counts at different thresholds
thresholds = [0.01, 0.05, 0.10]
print("FDR Threshold | BH Discoveries | Q-value Discoveries")
print("-" * 55)
for t in thresholds:
    _, padj_bh, _, _ = multipletests(pvals, alpha=t, method='fdr_bh')
    n_bh = np.sum(padj_bh <= t)
    n_qval = np.sum(qvals <= t)
    print(f"    {t:.2f}       |     {n_bh:5d}      |       {n_qval:5d}")

**Exercise 3.1**: How do the q-value discovery counts compare to BH? Why might they differ?

(Hint: q-values incorporate the $\pi_0$ estimate. When many genes are truly null, $\pi_0$ is close to 1 and the methods are similar. When many genes have signal, $\pi_0 < 1$ and q-values can be more powerful.)

**YOUR ANSWER:**

---

## Part 4: The Empirical Null (Simulation)

Efron's insight: sometimes the theoretical null $N(0,1)$ is wrong. When you convert p-values to z-scores, the actual null z-scores might be **wider** or **shifted** compared to the standard normal. This can happen due to unobserved confounders, correlations among genes, or other systematic effects.

Let's examine the z-score distribution from our real data.

In [ ]:
np.random.seed(2026)

# Convert p-values to two-sided, signed z-scores using sign(t_stats) from cell-2.
# Using sign(t_stats) is robust to whichever group is encoded as "ALL" first;
# using sign(log2fc) works here only because we defined log2fc with the same sign convention.
z_scores = stats.norm.ppf(1 - pvals / 2) * np.sign(t_stats)

# Remove any infinite z-scores (from p-values exactly 0 or 1)
finite_mask = np.isfinite(z_scores)
z_finite = z_scores[finite_mask]
print(f"Finite z-scores: {len(z_finite)} out of {len(z_scores)}")

# Plot histogram of z-scores with theoretical null overlay
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(z_finite, bins=120, density=True, color='steelblue', edgecolor='white',
        alpha=0.7, label='Observed z-scores')

z_grid = np.linspace(-6, 6, 300)
ax.plot(z_grid, stats.norm.pdf(z_grid, 0, 1), 'r-', linewidth=2.5,
        label='Theoretical null N(0, 1)')

# Empirical null: fit a normal to central z-scores (|z| < 2).
# CAVEAT: fitting an unrestricted Normal via MLE to the truncated subset systematically
# under-estimates sigma_0 (the tails outside |z|<2 are discarded). Efron's locfdr uses
# a truncated-normal MLE that corrects this; here we accept ~10-15% downward bias on
# sigma_0 for pedagogical clarity, and use a generous decision threshold below (1.2 / 0.8).
central_z = z_finite[np.abs(z_finite) < 2]
mu_emp, sigma_emp = stats.norm.fit(central_z)
ax.plot(z_grid, stats.norm.pdf(z_grid, mu_emp, sigma_emp), 'g--', linewidth=2.5,
        label=f'Empirical null N({mu_emp:.2f}, {sigma_emp:.2f}$^2$)')

ax.set_xlabel('z-score')
ax.set_ylabel('Density')
ax.set_title('Z-score Distribution: Theoretical vs. Empirical Null')
ax.legend(fontsize=11)
ax.set_xlim(-7, 7)
plt.tight_layout()
plt.show()

print(f"\nEmpirical null parameters:")
print(f"  Mean (delta_0):  {mu_emp:.4f}  (theoretical: 0)")
print(f"  SD (sigma_0):    {sigma_emp:.4f}  (theoretical: 1)")
# Efron's working rule: flag sigma_0 > ~1.2 as substantively wider than N(0,1).
if sigma_emp > 1.2:
    print(f"  --> sigma_0 > 1.2: the empirical null is substantively wider than N(0,1).")
    print(f"      Hidden variability (batch, correlation, latent structure) likely. Use empirical null.")
elif sigma_emp < 0.8:
    print(f"  --> sigma_0 < 0.8: the empirical null is narrower than N(0,1). Unusual; check test specification.")
else:
    print(f"  --> sigma_0 close to 1: the theoretical null N(0,1) is adequate for this dataset.")


**Exercise 4.1**: If $\sigma_0 > 1$ (the empirical null is wider than the theoretical), what happens if you use the theoretical null $N(0,1)$ instead? Would you get more or fewer discoveries? Would they be trustworthy?

**YOUR ANSWER:**


**Exercise 4.2 — Multi-turn: why is the empirical null wider than $N(0,1)$?** *(required multi-turn)*

This is the multi-turn AI exercise for Lab 2. **Three turns of model output in the same chat session** so the AI can build on (and revise) its earlier answer — *don't start a fresh conversation between turns*.

**Turn 1 — initial prompt.** Ask the AI:

> "I have $z$-scores from gene-level two-sample $t$-tests on a microarray experiment. When I fit a normal to the central peak of the $z$-score histogram (|z| < 2), I get $\hat\sigma_0 \approx 1.13$ — wider than $N(0,1)$. Why might the empirical null be wider than the theoretical $N(0,1)$? Two short paragraphs."

**AI's response (Turn 1, paste here):**


**Your critique (write your own — do not copy the bullets below verbatim).** Common failure modes to look for:

- The AI says "batch effects" or "correlation" but doesn't explain *how* those translate into a wider null.
- The AI does not distinguish between *unobserved covariates* (latent population structure) and *correlation among tests* (shared noise sources).
- The AI's answer is generic — it would apply to any "wider-than-expected null" anywhere in statistics — without engagement with the genomic context.
- The AI does not actually propose a diagnostic or fix; it just lists possible causes.

**YOUR CRITIQUE OF THE TURN-1 RESPONSE:**


**Turn 2 — your follow-up prompt.** Based on the specific weaknesses you flagged, write a prompt in your own words that asks the AI to revise its answer with the diagnostic protocol you actually need.

**YOUR TURN-2 PROMPT:**

**AI's response (Turn 2):**


**Turn 3 — apply the protocol to *this* notebook.** In the same chat session, ask the AI:

> "Now apply your protocol to my actual numbers: $\hat\sigma_0 \approx [PASTE YOURS]$, $\hat\delta_0 \approx [PASTE YOURS]$, and the threshold for 'substantively wider than $N(0,1)$' the lab uses is $\hat\sigma_0 > 1.2$. Should I use the empirical null or the theoretical null on this dataset, and what one line of code do I change to swap them?"

**YOUR TURN-3 PROMPT:**

**AI's response (Turn 3):**


**Final assessment.** Did the Turn-3 response give you a *concrete one-line code change* and an explicit recommendation, or did it remain hand-wavy? If still hand-wavy, write the right one-line change yourself.

---

## Part 5: Volcano Plot

A volcano plot shows fold change ($x$-axis) vs. statistical significance ($y$-axis) for every gene. It's the standard visualization for differential expression results.

In [ ]:
# Volcano plot: -log10(p-value) vs. log2 fold change
neg_log10_p = -np.log10(pvals)

# BH significance
reject_bh, padj_bh, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')

fig, ax = plt.subplots(figsize=(10, 7))

# Non-significant genes
ns_mask = ~reject_bh
ax.scatter(log2fc[ns_mask], neg_log10_p[ns_mask], 
           s=3, alpha=0.3, color='grey', label='Not significant')

# Significant genes
sig_mask = reject_bh
ax.scatter(log2fc[sig_mask], neg_log10_p[sig_mask], 
           s=8, alpha=0.6, color='crimson', label=f'BH significant (FDR < 0.05, n={np.sum(sig_mask)})')

# Label top 5 most significant genes
top5_idx = np.argsort(pvals)[:5]
for idx in top5_idx:
    ax.annotate(gene_names[idx], 
                xy=(log2fc[idx], neg_log10_p[idx]),
                xytext=(5, 5), textcoords='offset points',
                fontsize=9, fontweight='bold',
                arrowprops=dict(arrowstyle='-', color='black', lw=0.5))

# Uncorrected threshold
ax.axhline(y=-np.log10(0.05), color='blue', linestyle=':', alpha=0.5, label='p = 0.05 (uncorrected)')

# BH-adjusted threshold: the largest raw p-value among BH-rejected genes
if np.any(reject_bh):
    bh_cutoff_pval = np.max(pvals[reject_bh])
    ax.axhline(y=-np.log10(bh_cutoff_pval), color='green', linestyle='--', alpha=0.7,
               label=f'Effective discovery boundary on the raw p-scale (p = {bh_cutoff_pval:.4f})')

ax.set_xlabel('Mean Difference (ALL − AML)', fontsize=13)
ax.set_ylabel('$-\\log_{10}$(p-value)', fontsize=13)
ax.set_title('Volcano Plot: ALL vs. AML Leukemia', fontsize=14)
ax.legend(fontsize=10, loc='upper right')
plt.tight_layout()
plt.show()

---

## Part 6: Method Comparison

Let's bring all the methods together in one summary table.

In [ ]:
# Bonferroni
reject_bonf, _, _, _ = multipletests(pvals, alpha=0.05, method='bonferroni')

# BH
reject_bh, _, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')

# Q-values (using our manual computation)
reject_qval = qvals <= 0.05

# Summary table
summary = pd.DataFrame({
    'Method': ['Uncorrected (alpha = 0.05)', 
               'Bonferroni (FWER <= 0.05)', 
               'BH (FDR <= 0.05)', 
               'Q-value (FDR <= 0.05)'],
    'Discoveries': [
        np.sum(pvals < 0.05),
        np.sum(reject_bonf),
        np.sum(reject_bh),
        np.sum(reject_qval)
    ],
    'Notes': [
        'No correction — massively inflated',
        'Controls FWER — very conservative',
        'Controls FDR (BH bound uses pi0 = 1 implicitly — slightly conservative)',
        f'Controls FDR — uses pi0 = {pi0_hat:.3f}'
    ]
})

print(summary.to_string(index=False))

# Visual comparison
fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']
bars = ax.barh(summary['Method'], summary['Discoveries'], color=colors, edgecolor='white')
ax.set_xlabel('Number of Discoveries')
ax.set_title('Method Comparison: Discoveries at 5% Threshold')
for bar, val in zip(bars, summary['Discoveries']):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2, 
            str(val), va='center', fontweight='bold')
plt.tight_layout()
plt.show()

---

## Part 7: AI Extension — Local FDR

Efron's **local FDR** gives a gene-level probability of being null:

$$\text{fdr}(z) = \frac{\pi_0 f_0(z)}{f(z)}$$

where $f_0(z)$ is the null density and $f(z)$ is the overall mixture density. This is more informative than the "tail-area" FDR from BH, because it tells you about each individual gene rather than a set of genes.

**Your task**: ask an AI to implement local FDR in Python and critique the result.

**Suggested prompt** (paste-ready, with format directive):

> "Reply with a single Python code block (no prose, no .docx file, no surrounding explanation — just runnable code I can paste into a Colab cell). Given a numpy array `z_finite` of $z$-scores from gene-level Welch $t$-tests and a precomputed `pi0_hat` (Storey's estimate), implement Efron's local FDR: (1) estimate the mixture density $f(z)$ with `scipy.stats.gaussian_kde` using Silverman's rule; (2) take the null density $f_0(z) = \mathcal{N}(0, 1)$ via `scipy.stats.norm.pdf(z, 0, 1)`; (3) compute $\text{fdr}(z) = \pi_0 f_0(z) / f(z)$ for each gene and clip to $[0, 1]$; (4) plot $\text{fdr}(z)$ as a function of $z$ across a grid from $-6$ to $6$, with a horizontal line at the conventional 0.20 threshold and vertical lines at the $z$-values where the curve crosses 0.20. Use numpy, scipy.stats, and matplotlib only."

Paste the AI-generated code in the cell below and run it.

In [ ]:
# YOUR CODE HERE (AI-generated or your own)


### Critique Checklist

Paste a one-line summary of the AI's local-FDR plot (where does $\widehat{\mathrm{fdr}}(z)$ cross 0.20?), then complete the table.

**Summary of the AI's plot:**


| Criterion | Pass / Fail / Partial | Notes |
|---|---|---|
| Did it estimate $f(z)$ nonparametrically (e.g., `gaussian_kde`)? | | |
| Did it use $f_0(z) = N(0,1)$ from `scipy.stats.norm.pdf` (or the empirical null if specified)? | | |
| Did it multiply by $\hat\pi_0$ (using the variable from Part 2), or hard-code $\pi_0 = 1$? | | |
| Are the local-FDR values between 0 and 1 for all genes (clipped at 1)? | | |
| Does the curve make sense (high in the middle near $z = 0$, low in the tails)? | | |
| Did the plot mark the 0.20 threshold and the $z$-crossings? | | |

**One-sentence overall assessment** — was the AI's local-FDR implementation usable, or did you have to edit it?

---

## Wrap-Up Questions

1. Why does the BH procedure give different results from q-values? Which is more powerful, and under what condition?

**YOUR ANSWER:**

2. If the empirical null has $\sigma_0 > 1.2$, and you used $N(0,1)$ instead, would you get more or fewer discoveries? Would they be reliable?

**YOUR ANSWER:**

3. When would you report BH-adjusted p-values vs. q-values in a paper?

**YOUR ANSWER:**

4. Lecture 2 distinguished the *tail-area* FDR (a property of a rejection region) from the *local* FDR (a property of an individual test). Why is the local FDR more informative at the level of an individual gene that sits near the rejection boundary?

**YOUR ANSWER:**

---

## Looking Ahead

You now have a list of differentially expressed genes with FDR control. But you've only answered *which genes differ*. In **Lab 3**, you'll ask whether the samples have lower-dimensional structure (PCA, UMAP), and in **Lab 4**, you'll build a predictive gene signature using penalized regression.